https://www.reinforcement-learning.com/kb/q-learning

# Q-Learning from Scratch 🐣

I shall teach an agent to navigate a 4×4 grid:
- Reach the **goal** → +10 reward
- Fall into the **pit** → −10 reward
- Every other step → −0.1 (so it learns to be quick)

By the end, I shall have shown you how the agent learns a policy (best action per cell) **without being told the rules** — it figures them out from trial and error.

In [1]:
import numpy as np
import random

### Imports
I shall use:
- `numpy` → for the Q-table (a big array of numbers)
- `random` → for exploration (random moves)

In [3]:
GRID_SIZE = 4
START = (0, 0)
GOAL  = (3, 3)
PIT   = (1, 1)

# Actions: 0=up, 1=down, 2=left, 3=right
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
N_ACTIONS = len(ACTIONS)

def step(state, action):
    """Move the agent and return (next_state, reward, done)."""
    dr, dc = ACTIONS[action]
    r, c = state
    nr = min(max(r + dr, 0), GRID_SIZE - 1)   # clamp to grid edges
    nc = min(max(c + dc, 0), GRID_SIZE - 1)
    next_state = (nr, nc)

    if next_state == GOAL:
        return next_state, 10.0, True          # big win, episode over
    if next_state == PIT:
        return next_state, -10.0, True         # big loss, episode over
    return next_state, -0.1, False             # small step cost

### The Environment

I shall define `step(state, action)` as the "world". You give it where the agent is and
what it wants to do, and it tells you:

1. **next_state** — where the agent ends up (clamped to the grid so it can't walk off)
2. **reward** — what it earned
3. **done** — is the episode over? (yes at goal or pit)

This is the *only* thing the agent can interact with. It doesn't know
the rules — it just calls `step()` and observes.

In [4]:
# Q[state_row, state_col, action] = expected total future reward
Q = np.zeros((GRID_SIZE, GRID_SIZE, N_ACTIONS))
Q.shape

(4, 4, 4)

### The Q-Table — the agent's brain 🧠

I shall create a 3D array of shape `(4, 4, 4)`:
- 2 dimensions for the grid (row, column)
- 1 dimension for the 4 possible actions

So `Q[r, c, a]` = *"how much total reward do I expect if I'm at (r,c) and take action a?"*

Everything starts at **0** because the agent knows nothing yet.

In [5]:
alpha   = 0.1   # learning rate — how big each update step is
gamma   = 0.9   # discount factor — how much we value the future
epsilon = 0.2   # exploration rate — 20% random moves
EPISODES = 500  # how many full playthroughs to train on

### Hyperparameters — the knobs I shall tune

| Name | Meaning | Plain English |
|------|---------|---------------|
| `alpha` (α) | Learning rate | How much to trust the new estimate vs. the old one |
| `gamma` (γ) | Discount | 0.9 means "a reward 1 step ahead is worth 90% of one right now" |
| `epsilon` (ε) | Exploration | Probability of taking a random move instead of the best-known one |
| `EPISODES` | Training length | How many times I shall let the agent play the game |

These are **not learned** — *I* choose them. Try changing them later and see what happens!

In [6]:
for episode in range(EPISODES):
    state = START
    done = False

    while not done:
        r, c = state

        # --- (1) epsilon-greedy action choice ---
        if random.random() < epsilon:
            action = random.randint(0, N_ACTIONS - 1)   # explore
        else:
            action = int(np.argmax(Q[r, c]))             # exploit

        # --- (2) take the action, see what happens ---
        next_state, reward, done = step(state, action)
        nr, nc = next_state

        # --- (3) Q-learning update ---
        best_next = np.max(Q[nr, nc])
        td_target = reward + gamma * best_next * (0 if done else 1)
        Q[r, c, action] += alpha * (td_target - Q[r, c, action])

        state = next_state

print("Training done ✅")

Training done ✅


### The Training Loop — three steps per move

**Step 1 — Choose an action (ε-greedy)**
- With probability `epsilon` → **explore** (random move)
- Otherwise → **exploit** (pick the action with highest Q-value)

Why mix? Pure exploitation gets stuck. Pure exploration never uses what
it learned. I shall use the mix so the agent **discovers** good paths *and* **uses** them.

**Step 2 — Take the action**
Call `step()`, get `next_state`, `reward`, and `done`.

**Step 3 — Update the Q-value (the magic line) 📈**

$$Q(s,a) \leftarrow Q(s,a) + \alpha \big[\, r + \gamma \max_{a'} Q(s',a') - Q(s,a) \,\big]$$

Read it like a sentence:
> *"Move my estimate of `Q(s,a)` toward `reward + best future value`."*

- `best_next = max Q(next_state)` — the best the agent could do from where it landed
- `(0 if done else 1)` — if the episode ended, there's no future to add
- `td_target - Q[r,c,action]` is the **error** — how wrong our old estimate was
- I shall multiply by `alpha` so each update is a *gentle nudge*, not a jump

Repeat for 500 episodes → the table converges to the true optimal values.

In [8]:
print("Learned policy (arrows = best action per cell):\n")
arrows = "↑↓←→"
for r in range(GRID_SIZE):
    row = []
    for c in range(GRID_SIZE):
        if (r, c) == GOAL:
            row.append(" GOAL ")
        elif (r, c) == PIT:
            row.append(" PIT  ")
        else:
            best = int(np.argmax(Q[r, c]))
            row.append(f"  {arrows[best]}   ")
    print("".join(row))

Learned policy (arrows = best action per cell):

  →     →     →     ↓   
  ↑    PIT    →     ↓   
  →     →     →     ↓   
  →     →     →    GOAL 


### Reading the Policy

For each cell, I shall look at `Q[r, c]` — the row of 4 action-values —
and pick the `argmax` (the action with the biggest number).

That **best action** is the agent's policy: *"from this square, go this way."*

The arrows should form a path from **START (top-left)** to **GOAL
(bottom-right)** that avoids the **PIT**.

In [9]:
def show_path(episode_len=20):
    state = START
    path = [state]
    for _ in range(episode_len):
        if state == GOAL or state == PIT:
            break
        r, c = state
        action = int(np.argmax(Q[r, c]))   # pure greedy — no exploration
        state, _, done = step(state, action)
        path.append(state)
        if done:
            break
    return path

path = show_path()
print("Agent's route:", " → ".join(str(p) for p in path))

Agent's route: (0, 0) → (0, 1) → (0, 2) → (0, 3) → (1, 3) → (2, 3) → (3, 3)


### Following the Greedy Policy

Now that training is done, I shall set **ε = 0** (no exploration). The agent
just uses its Q-table: at each step, take the action with the largest Q-value.

The printed path is what the agent *actually does* when it "plays for real."
You should see it reach the GOAL without stepping on the PIT.

## Recap 🎓

1. **Q-table** = agent's memory of "how good is each action in each state"
2. **ε-greedy** = balance between trying new things and using what you know
3. **Update rule** = nudge estimates toward `reward + discounted future value`
4. **After enough episodes**, the table converges and greedy actions form an optimal path

That's the entire idea behind Q-learning — I shall have you note that everything else
(deep Q-networks, double Q-learning, etc.) is a refinement of these four ideas.